# Progressive train-validation grid search

This notebook uses only the published train and validation manifests. It persists every trial immediately, skips completed configuration hashes on rerun, and never loads test data. Run it on CUDA after the CPU smoke test in `04_train_model.ipynb` has passed.

In [ ]:
from dataclasses import replace
from pathlib import Path
import json

import pandas as pd
import torch

from reservoir_segmentation.training.config import ExperimentConfig, RuntimeConfig
from reservoir_segmentation.training.data import (
    build_split_datasets, fit_channel_statistics, write_channel_statistics,
)
from reservoir_segmentation.training.grid_search import (
    make_finalist_repetitions, make_progressive_baseline_configurations,
    make_weighted_configurations, run_grid_search,
)


In [ ]:
REPOSITORY_ROOT = next((path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (path / 'pyproject.toml').is_file()), None)
if REPOSITORY_ROOT is None:
    raise FileNotFoundError('Could not locate the repository root. Start Jupyter from the repository or its notebooks directory.')
GEOMETRY_ROOT = REPOSITORY_ROOT / 'data/processed/riverscope/geometry-v2'
SOURCE_PATH = REPOSITORY_ROOT / 'data/raw/RiverScope.zip'
ARTIFACT_ROOT = REPOSITORY_ROOT / 'artifacts/experiments/grid_search'
DEVICE = 'cuda'

if DEVICE == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('Full grid search requires CUDA. Use CPU only for tests and smoke tests.')

raw_datasets = build_split_datasets(GEOMETRY_ROOT, SOURCE_PATH)
statistics = fit_channel_statistics(raw_datasets['train'])
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
write_channel_statistics(statistics, ARTIFACT_ROOT / 'train_channel_statistics.json')
datasets = build_split_datasets(GEOMETRY_ROOT, SOURCE_PATH, statistics=statistics)
statistics.to_dict()


## Baseline CE: 12 architecture/optimizer trials

Only learning rate, base channels, and batch size vary. AdamW, 100 maximum epochs, scheduler, early stopping, and the ASSD-with-Dice checkpoint rule remain fixed.

In [ ]:
template = ExperimentConfig(runtime=RuntimeConfig(seed=42, device=DEVICE, num_workers=2))
baseline_configs = make_progressive_baseline_configurations(template)
baseline_records = run_grid_search(
    baseline_configs, datasets['train'], datasets['valid'],
    output_dir=ARTIFACT_ROOT / 'baseline',
    channel_statistics=statistics.to_dict(),
    repository_root=REPOSITORY_ROOT,
)
pd.DataFrame(baseline_records).sort_values(['state', 'selection_assd_macro_px'])


In [ ]:
baseline_winner_payload = json.loads((ARTIFACT_ROOT / 'baseline/winner.json').read_text())
baseline_winner = ExperimentConfig.from_dict(baseline_winner_payload['experiment_config'])
baseline_finalists = run_grid_search(
    make_finalist_repetitions(baseline_winner), datasets['train'], datasets['valid'],
    output_dir=ARTIFACT_ROOT / 'baseline_finalists',
    channel_statistics=statistics.to_dict(), repository_root=REPOSITORY_ROOT,
)
pd.DataFrame(baseline_finalists).sort_values('selection_assd_macro_px')


## Weighted CE: 8 loss trials, fixed architecture

The selected CE architecture and batch size remain fixed. Only `alpha`, `sigma`, and the allowed finalist LR adjustment vary, keeping CE and weighted CE comparable.

In [ ]:
weighted_configs = make_weighted_configurations(baseline_winner)
weighted_records = run_grid_search(
    weighted_configs, datasets['train'], datasets['valid'],
    output_dir=ARTIFACT_ROOT / 'weighted',
    channel_statistics=statistics.to_dict(), repository_root=REPOSITORY_ROOT,
)
pd.DataFrame(weighted_records).sort_values(['state', 'selection_assd_macro_px'])


In [ ]:
weighted_winner_payload = json.loads((ARTIFACT_ROOT / 'weighted/winner.json').read_text())
weighted_winner = ExperimentConfig.from_dict(weighted_winner_payload['experiment_config'])
weighted_finalists = run_grid_search(
    make_finalist_repetitions(weighted_winner), datasets['train'], datasets['valid'],
    output_dir=ARTIFACT_ROOT / 'weighted_finalists',
    channel_statistics=statistics.to_dict(), repository_root=REPOSITORY_ROOT,
)
pd.DataFrame(weighted_finalists).sort_values('selection_assd_macro_px')


Review the two three-seed finalist tables using validation metrics only. Freeze a final configuration and checkpoint for `04_train_model.ipynb`; do not inspect or optimize on test data here.